# 07 - Automated Pipeline

Runs stages 00 through 08 sequentially. Idempotent: skips stages whose artifacts already exist. Resumable if the session dies (just rerun this cell). Independent stages continue on failure; critical stages abort.

Controls:
- `!python scripts/run_pipeline.py` runs everything pending
- `--from 03_train_pc` resumes from a stage
- `--only 07_density` runs a single stage
- `--force` reruns even if the artifact exists

In [ ]:
import os, sys
REPO_DIR = '/kaggle/working/pc-sae'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
!python scripts/run_pipeline.py

## Pipeline status and consolidated results

In [ ]:
import pandas as pd
from src.utils import METRICS, LOG_DIR, load_json

status = load_json(LOG_DIR / 'pipeline_status.json')
print('started :', status.get('started'))
print('finished:', status.get('finished'))
print('\n=== STAGES ===')
for s in status['stages']:
    if s.get('skipped'):
        print(f"  {s['stage']:<14} SKIPPED")
    else:
        flag = 'OK' if s.get('ok') else 'FAIL'
        print(f"  {s['stage']:<14} {flag:<5} rc={s.get('returncode')} {s.get('duration_s')}s")

In [ ]:
print('=== RESULTS MASTER CSV ===')
pd.read_csv(METRICS / 'results_master.csv')

## Logs

In [ ]:
!cat outputs/logs/pipeline.log 2>/dev/null | tail -60
print('\n--- ERRORS ---')
!cat outputs/logs/errors.log 2>/dev/null | tail -30